# Worked solutions · Section 2 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F5.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Proposition 1.1.11 (`prop.decompositionOfLimitOrder`), eq. (1.1.1, `eq.orderingOfLimitOrders`).
Prices are integer counts of ticks.
A side of a book is written as its occupied levels, price $\times$ size, best first.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `02-updating-the-book` runs them and verifies every key.

In [ ]:
from math import log

# Section 2 · Updating the book

**F1. The book and three quantities derived from it.**
The book is the configuration of Section 1.1 (`sec.orderDrivenMarkets`):
the best bid $P^b$, the highest price holding buy orders;
the best ask $P^a$, the lowest price holding sell orders;
and the sizes $S^{b,i}$ at $P^b - (i-1)\tau$ and $S^{a,i}$ at $P^a + (i-1)\tau$, for $i = 1, 2, \dots$.
By the definitions of that section, the spread is $\phi = |P^a - P^b|$, which is $P^a - P^b$ since the sides never overlap,
the mid-price is $P^m = (P^a + P^b)/2$,
and the queue imbalance over the first $n$ levels is
$$I^n = \frac{\sum_{i\le n} S^{b,i} - \sum_{i\le n} S^{a,i}}{\sum_{i\le n} S^{b,i} + \sum_{i\le n} S^{a,i}},$$
eq. (1.1.2, `eq.queueImbalance`): bid minus ask, so that a bid-heavy book has $I^n > 0$.
In ticks, `book.spread` is $\phi$, `book.mid_price` is $P^m$,
and `book.best_bid_size` and `book.best_ask_size` are $S^{b,1}$ and $S^{a,1}$.

**F2. Matching.**
An incoming order $(t, q, p, d)$, with $d = +1$ for a buy and $d = -1$ for a sell,
matches a resting order $(s, \rho, \pi, -d)$ of the opposite side when $\pi d \le p d$:
a buy meets the asks at or below its limit, a sell the bids at or above it.
The two trade $\rho \wedge q$ shares at $\pi$, the price of the resting order
(the paragraph before eq. (1.1.1, `eq.orderingOfLimitOrders`)).
The search follows the priority of eq. (1.1.1, `eq.orderingOfLimitOrders`),
the better price first, $d\pi < d\pi'$:
a buy takes the lowest ask first, and a sell the highest bid first.

**F3. The decomposition.**
By Proposition 1.1.11 (`prop.decompositionOfLimitOrder`),
processing the sell $(t, q, p, -1)$ is processing its market part $(t, q_M, 0, -1)$
followed by its resting part $(t, q - q_M, p, -1)$, where
$$q_M = \min\Big(q, \sum_{i\geq 1} S^{b,i}\,\mathbf 1_{\{P^{b,i} \geq p\}}\Big),$$
the smaller of $q$ and the size at the eligible prices;
for a buy, the market part is $(t, q_M, \infty, +1)$ and
$q_M = \min\big(q, \sum_{i\geq 1} S^{a,i}\,\mathbf 1_{\{P^{a,i} \leq p\}}\big)$.
The submission branch of Algorithm 1.2.1 (`algo.bookFold`) is this decomposition:
it sets $r := q_M$, matches at the best eligible price while $r > 0$,
removing a price whose size reaches zero,
and adds $q - q_M$ at $p$.
It terminates because each pass empties one price or brings $r$ to zero
(the sentence after the algorithm).

**F4. The update of the package.**
Alternative A of 2.2 is `AggregateBook.submit`, Listing 1.2 (`listing.submit`), written as a function.
The update departs from Algorithm 1.2.1 (`algo.bookFold`) in three places,
described in the paragraphs after Table 1.2 (`tab.foldToCode`):
- its loop starts from $q$, not from $q_M$,
  and stops at the first best price that fails the test,
  written `best * d > p * d` so that one line serves both sides;
  the shares it has matched when it stops are $q_M$;
- it stops on an empty opposite side, `best is None`, as on a price that fails the test;
- the remainder rests at `resting_price(p, last)`:
  at $p$ for an order that names a price,
  and at the price of its last fill for a market order, the *market-to-limit* rule.
  A market order that matched nothing has no price to inherit, and its remainder rests nowhere.

A market order is a submission carrying a sentinel price,
0 for a sell and `sys.maxsize` for a buy in place of $\infty$
(Section 1.2.3 (`sec.foldImplementation`)),
and `is_market_price` recognises both.
Every write goes through `set_size`, which removes a price whose size is zero,
and `best_price` is `max` of the keys of the bids or `min` of the keys of the asks,
Listing 1.1 (`listing.aggregateStorage`).

**F5. The book of 2.2 to 2.7.**
Bid $1000 \times 100$, $999 \times 200$, $998 \times 150$; ask $1002 \times 120$, $1003 \times 180$.
The bid side holds 450 shares, 300 of them at 999 or above,
and the ask side holds 300.

## 2.1 · `question.limitBuyRestsInsideTheSpread`

Bid $1000 \times 300$, $999 \times 100$; ask $1001 \times 50$, $1002 \times 100$, $1004 \times 200$.
A buy limit order of 200 shares at 1003, applied by the package.

**Answer: (C).** `1 1003.5 -0.6`.

**The market part.**
The asks eligible to a buy at 1003 are those at or below it (F2), $1001 \times 50$ and $1002 \times 100$,
so $q_M = \min(200, 150) = 150$ (F3).
The order buys 50 at 1001 and 100 at 1002;
the next ask, 1004, is above its limit, and the loop stops there (F4).

**The resting part.**
The remaining $200 - 150 = 50$ shares rest at the price of the order, 1003 (F3, F4).
The book is then bid $1003 \times 50$, $1000 \times 300$, $999 \times 100$; ask $1004 \times 200$.

**The three quantities** (F1).
$\phi = 1004 - 1003 = 1$, $P^m = 1003.5$, and
$$I^1 = \frac{50 - 200}{50 + 200} = -0.6.$$

In [ ]:
bid, ask = 1003, 1004
min(200, 50 + 100), ask - bid, (ask + bid) / 2, (50 - 200) / (50 + 200)

**The other choices.**
- (A) `2 1003.0 -0.6`: the remainder rested at the last fill, 1002,
  by the market-to-limit rule, which applies to a market order only (F4):
  bid $1002 \times 50$ against ask $1004 \times 200$.
- (B) `1 1003.5 0.6`: the imbalance taken as ask minus bid;
  eq. (1.1.2, `eq.queueImbalance`) is bid minus ask.
- (D) `4 1002.0 0.2`: the remainder dropped:
  bid $1000 \times 300$ against ask $1004 \times 200$.
- (E) `4 1002.0 0.333`: the limit ignored,
  the order walking on to buy 50 of the 200 shares at 1004, as a market buy would:
  bid $1000 \times 300$ against ask $1004 \times 150$.

In [ ]:
1004 - 1002, (1004 + 1002) / 2, 1004 - 1000, (1004 + 1000) / 2, (300 - 200) / (300 + 200), round((300 - 150) / (300 + 150), 3)

## 2.2 · `question.precomputedMarketPartOnAWalkingSell`

The book of F5, and a sell limit order of 400 shares at 999.
A is the update of the package (F4);
B computes $q_M$ first and walks $q_M$ shares from the best bid, with no price test.

**Answer: (A).** Both alternatives print `[(1000, 100), (999, 200)]`, then `998 999 100`.

**A.**
For a sell $d = -1$, and `best * d > p * d` reads $P^b < 999$.
The loop takes the 100 shares at 1000 and the 200 at 999, and stops at 998, below the limit.
It has matched $q_M = 300$ shares, and 100 remain.
They rest at 999, the price of the order, on the ask side,
where 999 is below 1002 and becomes the best ask.

**B.**
The eligible bids are those at or above 999 (F2), holding $100 + 200 = 300$ shares,
so $q_M = \min(400, 300) = 300$ (F3).
By eq. (1.1.1, `eq.orderingOfLimitOrders`) the eligible bids are the best ones,
so the loop, taking $q_M$ shares from the best bid down,
never passes below 999 and needs no price test.
It takes 100 at 1000 and 200 at 999, and rests $q - q_M = 100$ at 999.

Both price the fills at the resting orders (F2),
and both leave bid $998 \times 150$; ask $999 \times 100$, $1002 \times 120$, $1003 \times 180$.

In [ ]:
q_m = min(400, 100 + 200)
q_m, 400 - q_m

**The other choices.**
- (B) B prints `[(1000, 100), (999, 200), (998, 100)]`, then `998 1002 120`:
  $q_M$ taken to be $q$, the eligible size ignored:
  B's loop, which has no price test, walks on to 998 and fills all 400 shares,
  and $q - q_M = 0$ rests.
  $q_M$ is the size at the eligible prices, 300 (F3).
- (C) Both print `[(999, 300)]`: the market part taken as one fill of $q_M$ shares at the limit price;
  a fill trades at the price of the resting order (F2), one fill per level.
- (D) B prints `[(1000, 100), (999, 200), (998, 50)]`, then `998 999 50`:
  B's test `price * d <= p * d` read without $d$, as `price <= p`.
  The eligible bids would be 999 and 998, $q_M = \min(400, 350) = 350$,
  and the walk from the best bid would take 50 at 998 and rest 50 at 999.
  For a sell the factor $d$ reverses the inequality (F2).

## 2.3 · `question.precomputedMarketPartRestsAtTheSentinel`

The book of F5, and Alternatives A and B of 2.2:
a market sell of 1000 shares, then a buy limit order of 50 shares at 997.

**Answer: (D).** A prints `[] 997 998`; B prints `[(0, 50)] None 0`.

**The market sell.**
It carries the sentinel price 0 (F4), at which every bid is eligible:
both alternatives sell the 450 shares of the bid side, and 550 remain.
- A's loop stops on the empty side (F4),
  and `resting_price(0, 998)` returns 998, the price of its last fill, by the market-to-limit rule.
  The ask side becomes $998 \times 550$, $1002 \times 120$, $1003 \times 180$.
- B computes $q_M = \min(1000, 450) = 450$ and rests $q - q_M = 550$ at $p$, which is 0.
  The ask side becomes $0 \times 550$, $1002 \times 120$, $1003 \times 180$.

In both the bid side is empty.
The sentinel is a price specification and not a point of the grid,
so the remainder of a market order cannot rest at it (Section 1.2.3 (`sec.foldImplementation`));
B rests it there all the same.

**The limit buy at 997.**
- In A the best ask, 998, is above 997:
  nothing trades, and the buy rests at 997. `[] 997 998`.
- In B the ask at 0 is eligible to a buy at 997 (F2),
  so $q_M = \min(50, 550) = 50$, and B buys 50 shares at 0, the price of the resting order.
  Nothing rests, the bid side stays empty, and 500 shares remain at 0. `[(0, 50)] None 0`.

In [ ]:
bid_side = 100 + 200 + 150
bid_side, 1000 - bid_side, min(50, 1000 - bid_side), 1000 - bid_side - 50

**The other choices.**
- (A) Both print `[] 997 998`: B taken to rest its remainder at its last fill, as A does;
  B rests it at $p$, the sentinel.
- (B) Both print `[] 997 1002`: the 550 unfilled shares taken to rest nowhere in both;
  the ask would stay at 1002, and the buy would rest at 997.
- (C) A prints `[] 997 1002`: A's remainder taken to rest nowhere;
  a market order rests nowhere only when it has matched nothing (F4).

## 2.4 · `question.eligibleBidsWalkedFromTheLowest`

The book of F5, and a sell limit order at 999, of 400 shares and then of 250, each on a book of its own.
The variant walks the eligible bids in the order that `sorted` gives them.

**Answer: (B).** `400 998 150 999`, then `250 1000 50 1002`.

**The eligible bids.**
For a sell at 999 the test $\pi d \le p d$ reads $\pi \ge 999$ (F2): the bids at 1000 and 999.
`sorted` returns them in increasing order, `[999, 1000]`,
whereas eq. (1.1.1, `eq.orderingOfLimitOrders`) gives a sell the highest bid first.

**400 shares.**
The variant takes the 200 shares at 999, then the 100 at 1000,
and rests the remaining 100 at 999.
Both eligible levels go whole, so the order of the walk leaves no trace:
bid $998 \times 150$, ask $999 \times 100$, as with A. `400 998 150 999`.

**250 shares.**
The variant takes the 200 shares at 999 and 50 of the 100 at 1000,
leaving bid $1000 \times 50$, $998 \times 150$ and the ask at 1002. `250 1000 50 1002`.
A takes the 100 shares at 1000 and 150 of the 200 at 999, and leaves $999 \times 50$:
the two trade 250 shares at 1000 and 999, in different amounts.

In [ ]:
250 - 200, 100 - (250 - 200)

**The other choices.**
- (A) What A prints: `sorted` taken to return the bids from the highest,
  as eq. (1.1.1, `eq.orderingOfLimitOrders`) orders them for a sell; it returns them in increasing order.
- (C) `400 1000 100 999`, then `250 1000 100 1002`:
  the test $\pi d \le p d$ read without $d$, as $\pi \le 999$:
  the eligible bids would be 999 and 998, walked from 998.
- (D) `RuntimeError`: the loop taken to iterate the dictionary that `set_size` shrinks;
  it iterates the list that `sorted` returns, built before the first write.

## 2.5 · `question.fillsPricedAtTheIncomingOrder`

The book of F5: a sell limit order of 250 shares at 999, then a market sell of 100,
applied by A and by a variant that records each fill at the price $p$ of the incoming order.

**Answer: (A).** A prints `998 100 350 349700`; the variant prints `998 100 350 249750`.

**The book and the volume.**
The price of a fill is read again only by line 13, `last = fills[-1][0] ...`,
which places a remainder, and neither order leaves one.
The limit sell takes the 100 shares at 1000 and 150 of the 200 at 999;
the market sell, at the sentinel 0 (F4), takes the other 50 at 999 and 50 of the 150 at 998.
Both alternatives leave the best bid at $998 \times 100$, and trade $250 + 100 = 350$ shares.

**The value.**
A records the prices of the resting orders (F2):
$$100 \cdot 1000 + 200 \cdot 999 + 50 \cdot 998 = 349\,700.$$
The variant records 999 for the 250 shares of the limit sell
and the sentinel 0 for the 100 shares of the market sell: $250 \cdot 999 = 249\,750$.

In [ ]:
100 * 1000 + 200 * 999 + 50 * 998, 250 * 999 + 100 * 0

**The other choices.**
- (B) Both print `998 100 350 349700`: the variant taken to record the prices of the resting orders.
- (C) The variant prints `998 100 350 349550`:
  the market sell taken to carry the price of its last fill, 998,
  the market-to-limit rule (F4) read as pricing the order.
  At 998 its test admits the same bids, 999 and 998, so the walk is A's,
  and the variant records $250 \cdot 999 + 100 \cdot 998$.
- (D) A prints `999 50 250 249850` and the variant `999 50 250 249750`:
  the market sell taken to carry `sys.maxsize`, the sentinel of a buy.
  For a sell, `best * d > p * d` then holds at every bid:
  the market sell trades nothing, and with no fill to inherit it rests nowhere (F4).
  Only the limit sell trades: A records $100 \cdot 1000 + 150 \cdot 999$, the variant $250 \cdot 999$.

In [ ]:
250 * 999 + 100 * 998, 100 * 1000 + 150 * 999, 250 * 999

## 2.6 · `question.priceTestWithoutTheDirection`

The book of F5, each message applied to its own copy.
The variant stops its loop on `best > p`, in place of `best * d > p * d`.

**Answer: (C).** After (i) and (ii).

**The two tests.**
A stops at a best price that is not eligible, $\pi d > p d$ (F2, F4).
For a buy, $d = 1$, and the two tests are one.
For a sell, $\pi d > p d$ reads $\pi < p$, a bid below the limit,
whereas `best > p` stops at a bid above the limit, where the order should trade,
and lets it trade at a bid below.

- (i) A buy of 200 at 1002: the same test.
  Both buy 120 at 1002, stop at 1003, and rest 80 at 1002. Same book.
- (ii) A sell of 50 at 1000: the best bid equals the limit,
  and `1000 > 1000` is false, as is A's test.
  Both sell 50 at 1000, and the order is filled there,
  before the variant meets a price at which the two tests differ. Same book.
- (iii) A market sell of 50, at the sentinel 0: A sells 50 at 1000.
  The variant stops at once, `1000 > 0`;
  it has no fill whose price to inherit, and its 50 shares rest nowhere (F4).
  The book is unchanged. Different books.
- (iv) A sell of 50 at 1001: A stops at once, the bid 1000 being below the limit,
  and rests 50 at 1001.
  The variant passes `1000 > 1001` and sells 50 at 1000, below the limit. Different books.

**The other choices.**
- (A) After (i) only: the four messages taken to be applied in turn to one book.
  After (i) the best bid is $1002 \times 80$; A sells (ii) there,
  the variant stops at `1002 > 1000`, and the books differ from then on.
  Each message meets its own copy of the book.
- (B) After (i), (ii) and (iii): the market sell taken to pass any price test, as an order without a price.
  The sentinel 0 is a price to the test, and `1000 > 0` stops the variant.
- (D) After (i), (ii) and (iv): A's test read as `best * d > p`, the factor $d$ on one side only.
  For a sell `-best > p` never holds, so A, like the variant, sells (iv) at 1000.
  The factor $d$ multiplies both sides (F4).
- (E) After all four: `best > p` taken for A's test on both sides;
  the factor $d$ reverses the inequality for a sell.

## 2.7 · `question.splittingAMarketOrderLosesItsRemainder`

The book of F5. Each message is applied whole to one book,
and as its two parts to another, `split` computing $q_M$ on the book they meet.
A message of no shares, or at the price 0, is accepted.

**Answer: (B).** `True`, `False`, `True`.

**A sell of 400 at 999.**
$q_M = \min(400, 300) = 300$ (F3, F5): a market sell of 300 and a sell of 100 at 999.
The market part takes 100 at 1000 and 200 at 999;
the resting part meets the bid at 998, below its limit, and rests at 999.
The whole order does the same, as in 2.2. `True`.

**A market sell of 1000.**
$q_M = \min(1000, 450) = 450$: a market sell of 450, and a limit sell of 550 at the price 0.
The whole order sells the 450 shares of the bid side and rests 550 at 998, its last fill,
by the market-to-limit rule (F4).
In the parts, the market sell of 450 empties the bid side.
The limit sell at 0 carries the sentinel of a market sell, which `is_market_price` recognises,
and it meets an empty side:
it has no fill whose price to inherit, and rests nowhere (F4).
The asks differ at 998. `False`.

**A market buy of 150.**
$q_M = \min(150, 300) = 150$: a market buy of 150, and a limit buy of 0 shares at `sys.maxsize`.
The market part buys 120 at 1002 and 30 at 1003, as the whole order does.
The resting part has no shares:
its loop does not run, and `remaining > 0` is false, so it writes nothing. `True`.

In [ ]:
min(400, 100 + 200), min(1000, 100 + 200 + 150), 1000 - 450, min(150, 120 + 180), 150 - 120

**The other choices.**
- (A) `True` three times: the resting part taken to inherit 998, the last fill of the market part,
  as though the market-to-limit rule spanned the two messages.
  The remainder is a separate message at the sentinel 0 that matches nothing,
  so it has no fill whose price to inherit, and rests nowhere (F4).
- (C) `False`, `False`, `True`: the two parts applied in the reverse order, the resting part first.
  The sell at 999 then takes the 100 shares at 1000 and leaves nothing to rest,
  and the sell of 550 at 0 sells the whole bid side and rests 100 at 998, its last fill.
  Proposition 1.1.11 (`prop.decompositionOfLimitOrder`) applies the market part first.
- (D) `True`, `False`, `ValueError`: a message of no shares taken to be refused;
  a message is refused only when its size or its price is negative.

## 2.8 · `question.setSizeKeepingEmptyLevelsNeverReturns`

Bid $1000 \times 100$, $999 \times 200$; ask $1002 \times 120$, $1003 \times 180$.
A market buy of 120, then of 150, each on a book of its own,
in a subclass whose `set_size` keeps a price whose size reaches zero.

**Answer: (D).** It prints `1002 0 2`, and the second call of `apply` never returns.

**The override.**
`submit` is inherited from `AggregateBook`,
and its call `self.set_size` is resolved on the class of the book:
it calls the subclass's `set_size`, Proposition B.7.1 (`prop.lateBinding`).
`best_price` is inherited unchanged,
and returns `min` of the keys of the asks, whatever their sizes (F4).

**The buy of 120.**
At the sentinel `sys.maxsize` every ask is eligible.
The loop buys the 120 shares at 1002 and writes 0 there, which the subclass keeps;
`remaining` is 0, and the loop ends.
`best_ask_price` is `min` of the keys, still 1002;
`best_ask_size` reads the size there, 0;
and $\phi = 1002 - 1000 = 2$. `1002 0 2`.

**The buy of 150.**
The first pass buys 120 at 1002 and leaves 30 shares.
Every further pass finds the best ask at 1002, trades `min(30, 0)` shares, which is none,
and writes 0 back: it changes neither the book nor `remaining`,
and nothing else the loop reads changes, so the same pass repeats for ever.
Algorithm 1.2.1 (`algo.bookFold`) terminates because each pass empties one price or brings $r$ to zero (F3);
with the price kept at size zero, a pass here does neither.

In [ ]:
150 - 120, min(150 - 120, 0), 1002 - 1000

**The other choices.**
- (A) `1003 180 3`, then `1003 150 3`: the `set_size` of the package, which removes 1002,
  taken to be the one `submit` calls;
  it calls the override, Proposition B.7.1 (`prop.lateBinding`).
- (B) `1002 0 2` twice: the loop taken to walk on to 1003 after 1002, as a walk over the sorted prices would,
  leaving $1002 \times 0$ and $1003 \times 150$;
  it asks `best_price` on every pass, and `min` returns 1002.
- (C) `1002 0 2`, then `AssertionError`:
  `check_invariants`, which refuses a price kept at size zero, taken to run at the start of every pass:
  the buy of 120 ends after one pass, and the buy of 150 would raise at its second.
  The book calls it only in `from_levels`, where every size is positive.

## 2.9 · `question.withdrawalBeyondTheRestingSize`

Bid $1000 \times 100$, $999 \times 200$; ask $1002 \times 120$, $1003 \times 180$.
A market sell of 30, then the withdrawal of 100 shares at 1000 on the bid side,
on a lenient book and on a strict one.

**Answer: (A).** With `strict = False` it prints `999 3 1000.5`;
with `strict = True` it raises `ValueError`.

**The fill.** The market sell takes 30 of the 100 shares at 1000, and 70 remain.

**The withdrawal.**
`withdraw` is the cancellation branch of Algorithm 1.2.1 (`algo.bookFold`).
It removes at most the size resting at its price,
since a recorded feed begins with withdrawals of orders posted before the file did,
and a book built with `strict=True` raises instead
(the paragraph on `withdraw` in Section 1.2.3 (`sec.foldImplementation`)).
- Lenient: it removes `min(100, 70)` = 70 shares, `set_size` removes the price 1000,
  and the best bid falls to 999:
  $\phi = 1002 - 999 = 3$ and $P^m = 1000.5$ (F1).
- Strict: $100 > 70$, and `withdraw` raises `ValueError` before it writes anything.

In [ ]:
100 - 30, min(100, 100 - 30), 1002 - 999, (1002 + 999) / 2

**The other choices.**
- (B) `1000 2 1001.0` with `strict = False`:
  the lenient book taken to skip a withdrawal it cannot complete; it removes what rests.
- (C) `999 3 1000.5` in both runs: the fill forgotten;
  the withdrawal of 100 would then meet 100 shares, and the strict book would accept it.
- (D) `ValueError` in both runs: the lenient book taken to refuse as well;
  `withdraw` raises only when `self.strict` holds.

In [ ]:
1002 - 1000, (1002 + 1000) / 2

## 2.10 · `question.sweepRescanningTheBestPrice`

A market sell of $\ell$ shares against a bid side of $\ell$ prices holding one share each,
for $\ell = 10^3$ and $\ell = 10^4$.
A asks `best_price` for the best bid on every pass;
B sorts the eligible prices once and walks them.

**Answer: (C).** The time of A grows about a hundredfold, and that of B about tenfold.

**A.**
Each pass sells the one share at the best bid and removes that price,
so pass $k$ meets a side of $\ell - k + 1$ prices.
`best_price` is `max` of the keys of the side (F4),
a scan of $O(\ell)$, Table 1.4 (`tab.fasterBookCosts`),
and the $\ell$ passes scan $\ell + (\ell - 1) + \dots + 1 = \ell(\ell + 1)/2$ keys in all.
The rest of a pass costs $O(1)$ on average: a write, Table 1.4 (`tab.fasterBookCosts`),
and a read, a property of Python's `dict`.
From $\ell = 10^3$ to $\ell = 10^4$ the scans grow a hundredfold, and they dominate.

**B.**
It sorts the $\ell$ eligible prices once, in $O(\ell \log \ell)$ comparisons at most, a property of Python's `sorted`,
and then visits each price once, at $O(1)$ a visit.
From $\ell = 10^3$ to $\ell = 10^4$ its count of operations grows by a factor between 10 and $10\log 10^4/\log 10^3 \approx 13$,
and its time about tenfold.

In [ ]:
small, large = 10**3, 10**4
large * (large + 1) / (small * (small + 1)), large / small, large * log(large) / (small * log(small))

**The other choices.**
- (A) B slower: one sort compared with one scan; A scans on each of its $\ell$ passes.
- (B) Both tenfold, A slower by a constant factor:
  a best price taken to cost $O(1)$, as in `TickArrayBook`;
  in `AggregateBook` it costs $O(\ell)$, Table 1.4 (`tab.fasterBookCosts`).
- (D) Both a hundredfold: B's sort taken to be repeated on every pass, as A's scan is;
  it runs once, before the loop.

## 2.11 · `question.idealisedLimitBuyAtAFullTouch`

Bid $1000 \times 4$, $999 \times 5$, $998 \times 5$; ask $1001 \times 1$, $1002 \times 5$, $1003 \times 5$.
Three limit buys of one share each, placed by `_place` with $\bar S = 5$ and applied in turn.

**Answer: (B).** `[1000, 1001, 1001] 1001 1 1002 5`.

**The placement.**
Under Example 1.4.10 (`example.idealisedPlacement`) a limit order joins the best quote of its side,
or, once the size there has reached $\bar S$, arrives one tick above it (below, for a sell),
where it trades if the spread is one tick.
`_place` writes this rule for a limit buy:
it joins the touch while `book.size_at(direction, touch) < self.uniform_depth`;
otherwise it goes to `touch + 1`,
unless that price is the best ask, in which case it goes to the best ask through `_consumable_touch`.

**The first buy.**
The bid holds $4 < 5$: the share joins 1000, which then holds 5.

**The second buy.**
The bid holds 5, and the price inside it, 1001, is the best ask.
`_consumable_touch` raises only when the touch holds one share and the price behind it none;
1001 holds one share and 1002 holds five, so it returns 1001.
The buy at 1001 trades the one share there, as a market buy would
(the second paragraph after Assumption 1.4.1 (`assumption.idealisedBook`)),
and the best ask moves to $1002 \times 5$. The bid is unchanged.

**The third buy.**
The bid still holds 5 at 1000.
The price inside it, 1001, now lies inside a spread of two ticks:
the buy rests there and opens a new best bid, $1001 \times 1$.

**The other choices.**
- (A) `[1000, 1000, 1001] 1000 6 1002 5`: the test `<` read as `<=`,
  so that a touch holding $\bar S$ is still joined:
  the second share joins 1000, which then holds 6, and the third trades at 1001.
- (C) `[1000, 1001, 1002] 1000 5 1002 4`:
  a buy at a full touch taken always to trade at the best ask;
  it does so only when the spread is one tick, and the third buy meets a spread of two.
- (D) `AssumptionBreach` at the second event:
  the purchase of the one share at 1001 taken to empty the ask side;
  1002, behind it, holds five.